In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ==============================
# 1. 固定随机种子
# ==============================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==============================
# 2. 超参数
# ==============================
BATCH_SIZE = 128
EPOCHS = 8
LR = 1e-3
LAMBDA_REG = 1e-5   # 可以改成 1e-4 / 5e-5 试试
NUM_WORKERS = 0

# ==============================
# 3. 数据集
# ==============================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

# ==============================
# 4. CNN 模型
# ==============================
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28x28 -> 14x14

            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 14x14 -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(2)                               # 14x14 -> 7x7
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print("Model structure:\n")
print(SimpleCNN())

# ==============================
# 5. L1 正则项
# ==============================
def l1_regularization(model):
    l1 = torch.tensor(0.0, device=device)
    for name, param in model.named_parameters():
        if "weight" in name:
            l1 = l1 + torch.sum(torch.abs(param))
    return l1

# ==============================
# 6. 评估函数
# ==============================
def evaluate(model, dataloader, criterion, reg_type="none", lambda_reg=0.0):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if reg_type == "l1":
                # 评估时如果你想比较“总目标函数”，就加上 L1 项
                # 如果只想看纯分类损失，可以删掉下面两行
                loss = loss + lambda_reg * l1_regularization(model)

            total_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

# ==============================
# 7. 训练函数
# ==============================
def train_model(reg_type="none", lambda_reg=1e-5, epochs=8, lr=1e-3):
    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()

    if reg_type == "l2":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=lambda_reg)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if reg_type == "l1":
                loss = loss + lambda_reg * l1_regularization(model)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        test_loss, test_acc = evaluate(
            model, test_loader, criterion, reg_type=reg_type, lambda_reg=lambda_reg
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(
            f"[{reg_type.upper():^4}] "
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
        )

    return model, history

# ==============================
# 8. 训练三组模型
# ==============================
results = {}

for reg in ["none", "l1", "l2"]:
    print(f"\nTraining: {reg.upper()}")
    model, history = train_model(
        reg_type=reg,
        lambda_reg=LAMBDA_REG,
        epochs=EPOCHS,
        lr=LR
    )
    results[reg] = {
        "model": model,
        "history": history
    }

# ==============================
# 9. 画 Loss 曲线
# ==============================
plt.figure(figsize=(10, 6))
for reg, label in zip(["none", "l1", "l2"], ["No Reg", "L1", "L2"]):
    plt.plot(results[reg]["history"]["train_loss"], label=f"{label} Train Loss")
    plt.plot(results[reg]["history"]["test_loss"], linestyle="--", label=f"{label} Test Loss")
plt.title("Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

# ==============================
# 10. 画 Accuracy 曲线
# ==============================
plt.figure(figsize=(10, 6))
for reg, label in zip(["none", "l1", "l2"], ["No Reg", "L1", "L2"]):
    plt.plot(results[reg]["history"]["train_acc"], label=f"{label} Train Acc")
    plt.plot(results[reg]["history"]["test_acc"], linestyle="--", label=f"{label} Test Acc")
plt.title("Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# ==============================
# 11. 提取权重并画参数分布
# ==============================
def get_all_weights(model):
    weights = []
    for name, param in model.named_parameters():
        if "weight" in name:
            weights.extend(param.detach().cpu().numpy().ravel())
    return np.array(weights)

plt.figure(figsize=(15, 4))
for i, reg in enumerate(["none", "l1", "l2"], 1):
    plt.subplot(1, 3, i)
    w = get_all_weights(results[reg]["model"])
    plt.hist(np.abs(w), bins=40, alpha=0.8)
    plt.title(f"{reg.upper()} | Weight Magnitude")
    plt.xlabel("|w|")
    plt.ylabel("Count")
plt.tight_layout()
plt.show()

# ==============================
# 12. 随机可视化预测结果
# ==============================
classes = list(range(10))

def show_predictions(model, dataset, title, num_images=12):
    model.eval()
    indices = np.random.choice(len(dataset), num_images, replace=False)

    plt.figure(figsize=(12, 6))
    for i, idx in enumerate(indices, 1):
        image, label = dataset[idx]
        input_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            pred = output.argmax(dim=1).item()

        img = image.squeeze().cpu().numpy()

        plt.subplot(3, 4, i)
        plt.imshow(img, cmap="gray")
        plt.title(f"GT: {label} / Pred: {pred}")
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

for reg in ["none", "l1", "l2"]:
    show_predictions(results[reg]["model"], test_dataset, f"Predictions - {reg.upper()}")

# ==============================
# 13. 最终结果汇总
# ==============================
print("\nFinal Summary")
print("-" * 70)
for reg in ["none", "l1", "l2"]:
    final_train_acc = results[reg]["history"]["train_acc"][-1]
    final_test_acc = results[reg]["history"]["test_acc"][-1]
    w = get_all_weights(results[reg]["model"])
    print(
        f"{reg.upper():>4} | "
        f"Train Acc: {final_train_acc:.4f} | "
        f"Test Acc: {final_test_acc:.4f} | "
        f"Mean |w|: {np.mean(np.abs(w)):.6f} | "
        f"Near Zero Ratio (|w|<1e-3): {(np.abs(w) < 1e-3).mean():.4f}"
    )

Using device: cpu


100%|██████████| 9.91M/9.91M [00:02<00:00, 3.44MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 116kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.08MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.16MB/s]


Model structure:

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1568, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

Training: NONE
[NONE] Epoch 1/8 | Train Loss: 0.2090 | Train Acc: 0.9388 | Test Loss: 0.0577 | Test Acc: 0.9809


KeyboardInterrupt: 